# Lesson 01 - Panimula sa mga AI Agent

Maligayang pagdating sa unang leksyon sa kursong **AI Agents para sa mga Nagsisimula**!

Ang isang **AI agent** ay isang programa na gumagamit ng malaking language model (LLM) bilang makina ng pangangatwiran nito at maaaring gumawa ng *mga aksyon* sa tunay na mundo — pagtawag sa mga API, pagkuha ng datos mula sa mga database, o pagpapatakbo ng code — upang makamit ang isang layunin para sa isang gumagamit.

Sa notebook na ito ay bubuuin mo ang iyong unang agent: isang **Travel Agent** na nagrerekomenda ng mga destinasyon ng bakasyon. Sa daan ay matututunan mo kung paano:

1. Kumonekta sa Azure AI Foundry Agent Service gamit ang **Microsoft Agent Framework**.
2. Bigyan ang agent ng isang **tool** — isang payak na Python function na maaari nitong tawagan.
3. Patakbuhin ang agent at inspeksyunin ang tugon nito.
4. I-stream ang tugon ng agent token-by-token.


## Setup

Bago patakbuhin ang notebook na ito, siguraduhing mayroon kang:

1. **Isang Azure AI Foundry na proyekto** na may na-deploy na chat model (hal. `gpt-4o-mini`).
2. **Nakalog-in gamit ang Azure CLI** — patakbuhin ang `az login` sa iyong terminal.
3. **Na-set ang mga kinakailangang environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ang endpoint ng iyong Azure AI Foundry na proyekto.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ang pangalan ng iyong na-deploy na modelo.

Ini-install ng cell sa ibaba ang mga Python package na kailangan mo.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Paglikha ng Iyong Unang Ahente

Kailangan ng isang ahente ng dalawang bagay:

- **Mga Tagubilin** na nagsasabi sa kanya *kung sino siya* at *paano kumilos* (isang system prompt).
- **Mga Kagamitan** — Mga function sa Python na dinidekorahan ng `@tool` na maaaring tawagan ng ahente para kumuha ng impormasyon o magsagawa ng mga aksyon.

Sa ibaba ay nagtatakda kami ng isang simpleng kagamitan na nagbabalik ng listahan ng mga popular na destinasyon para sa bakasyon. Gagamitin ng ahente ang kagamitang ito kapag ang isang gumagamit ay humiling ng mga rekomendasyon sa paglalakbay.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Para sa mas interaktibong karanasan maaari mong i-**stream** ang tugon ng ahente. Sa halip na maghintay ng buong sagot, ang ahente ay nagbibigay ng mga piraso ng teksto habang ito ay ginagawa. Ito ay lalong kapaki-pakinabang sa mga chat interface kung saan nais mong ipakita ang output nang real time.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

Sa araling ito natutunan mo kung paano:

- **Gumawa ng provider** na kumokonekta sa Azure AI Foundry Agent Service gamit ang `AzureAIProjectAgentProvider`.
- **Mag-defina ng tool** gamit ang `@tool` decorator upang ang agent ay makatawag ng iyong mga Python function.
- **Patakbuhin ang agent** na may mensahe mula sa user at i-print ang tugon nito.
- **Mag-stream ng mga tugon** para sa output na real-time.

Sa susunod na aralin susuriin natin nang mas malalim ang mga agentic framework at matutunan kung paano bigyan ang mga agent ng mas makapangyarihang tools at kakayahan sa multi-step reasoning.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
